# Decoupled head

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/12-decoupled-head/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍與後續實驗〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.4.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""Separate branch gradients while retaining a shared backbone."""
import torch
from torch import nn
from torch.nn import functional as F


class DecoupledHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Conv2d(3, 8, 3, padding=1)
        self.box_branch = nn.Sequential(nn.Conv2d(8, 8, 3, padding=1), nn.ReLU(), nn.Conv2d(8, 4, 1))
        self.class_branch = nn.Sequential(nn.Conv2d(8, 8, 3, padding=1), nn.ReLU(), nn.Conv2d(8, 2, 1))

    def forward(self, x):
        f = F.relu(self.backbone(x))
        return self.box_branch(f), self.class_branch(f)


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    model = DecoupledHead()
    x = torch.rand(2, 3, 4, 4)
    distances = torch.full((2, 4, 4, 4), 1.5)
    classes = torch.zeros(2, 2, 4, 4)
    classes[:, 0] = 1  # independent sigmoid classes; simplified all-positive example
    boxes, logits = model(x)
    box_loss = F.smooth_l1_loss(F.softplus(boxes), distances)
    cls_loss = F.binary_cross_entropy_with_logits(logits, classes)
    model.zero_grad()
    box_loss.backward(retain_graph=True)
    assert model.box_branch[0].weight.grad is not None
    assert model.class_branch[0].weight.grad is None
    box_backbone_grad = model.backbone.weight.grad.clone()
    model.zero_grad()
    cls_loss.backward(retain_graph=True)
    assert model.box_branch[0].weight.grad is None
    class_backbone_grad = model.backbone.weight.grad.clone()
    cosine = F.cosine_similarity(box_backbone_grad.flatten(), class_backbone_grad.flatten(), dim=0)
    optimizer = torch.optim.SGD(model.parameters(), lr=.1)
    model.zero_grad()
    (box_loss + cls_loss).backward()
    assert torch.allclose(model.backbone.weight.grad, box_backbone_grad + class_backbone_grad, atol=1e-7)
    old = model.backbone.weight.detach().clone()
    optimizer.step()
    assert not torch.equal(old, model.backbone.weight)
    print('box / class shapes:', tuple(boxes.shape), tuple(logits.shape))
    print('classification-only backward leaves box branch grad=None')
    print(f'shared-backbone gradient cosine: {cosine.item():.4f}')
    print('total backbone gradient = box gradient + class gradient: verified')
    print('parameters:', sum(p.numel() for p in model.parameters()))


if __name__ == '__main__':
    main()


box / class shapes: (2, 4, 4, 4) (2, 2, 4, 4)
classification-only backward leaves box branch grad=None
shared-backbone gradient cosine: -0.0073
total backbone gradient = box gradient + class gradient: verified
parameters: 1446
